# Registrar modelo de predicción

In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

model_full_name = f"{catalog}.{schema}.{model_name}"
xp_name = f"{model_name}"
xp_path = f"{project_path}/Experiments"
model_alias = "Challenger" 
metric_score = "f1"
feature_table_name = f"{catalog}.{schema}.churn_user_features"

In [0]:
from mlflow.tracking import MlflowClient
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (EndpointCoreConfig, ServedEntityInput)

# 1. Inicializar cliente
mlflow_client = MlflowClient(registry_uri="databricks-uc")

# Get last model version
def get_last_model_version(model_name):
    versions = mlflow_client.search_model_versions(f"name='{model_name}'")
    latest = max(versions, key=lambda v: int(v.version))
    return latest

def get_experiment_path(experiment_name):
    experiment = mlflow_client.get_experiment_by_name(experiment_name)
    return experiment.experiment_id

def get_experiment(experiment_name):
    experiment = mlflow_client.get_experiment_by_name(experiment_name)
    return experiment.experiment_id

def exists_endpoint(name):
    return name in w.serving.list_endpoints()[0].name

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow_client = MlflowClient(registry_uri="databricks-uc")

latest_model = get_last_model_version(model_full_name)
runs = mlflow.search_runs(experiment_names=[xp_name], order_by=[f"metics.{score_metric} DESC"], 
                                filter_string="status = 'FINISHED'", max_results=1)
if runs.empty:
  raise Exception("No runs found")  

run_id = runs.iloc[0].run_id
latest_model = mlflow.register_model(f"runs:/{run_id}/model", f"{model_full_name}")

model_desc = f"Predecir probabilidad de abandono de clientes utililiza caracteristicas de las tablas: {feature_table_name}. Metrica para validar modelo: {metric_score}."

mlflow_client.update_registered_model(
  name=latest_model.name,
  version=latest_model.version,
  description=model_desc
)